# Modular Fair-RAG Recreation (T5 Small + BM25)

This notebook is designed to scale across machines by changing only one config profile.

Profiles:
- `weak`: 5 queries, small samples
- `balanced`: more queries and samples
- `strong`: intended for a better machine/full recreation

Output focus: normalized EE-D interval buckets (`0.0-0.2 ... 0.8-1.0`) and EU differences.

In [6]:
from pathlib import Path
import importlib
import pandas as pd

import notebook_experiment_utils as neu
importlib.reload(neu)

ExperimentConfig = neu.ExperimentConfig
find_repo_root = neu.find_repo_root
resolve_python_executable = neu.resolve_python_executable
ensure_paths_exist = neu.ensure_paths_exist
run_experiment_for_alpha = neu.run_experiment_for_alpha
run_gold_baseline = neu.run_gold_baseline
run_retriever_grid = neu.run_retriever_grid
run_deterministic_reference = neu.run_deterministic_reference
run_mmr_deterministic = neu.run_mmr_deterministic
normalize_eu_grid = neu.normalize_eu_grid
normalize_eu_for_retriever = neu.normalize_eu_for_retriever
load_normalized_rows = neu.load_normalized_rows
load_normalized_rows_for_retriever = neu.load_normalized_rows_for_retriever
load_raw_rows = neu.load_raw_rows
add_ee_d_interval_bins = neu.add_ee_d_interval_bins
summarize_by_interval = neu.summarize_by_interval
interval_pvalues_vs_deterministic = neu.interval_pvalues_vs_deterministic
save_interval_outputs = neu.save_interval_outputs
save_per_query_log = neu.save_per_query_log
save_pvalue_outputs = neu.save_pvalue_outputs
reset_run_outputs = neu.reset_run_outputs
assert_consistent_qids_across_alphas = neu.assert_consistent_qids_across_alphas


In [ ]:
ROOT = find_repo_root()
PY = resolve_python_executable(ROOT)

profiles = {
    'weak': {
        'max_queries': 30,
        'n_samples': 10,
        'k': 5,
        'print_interval': 10,
    },
    'balanced': {
        'max_queries': 833,
        'n_samples': 12,
        'k': 5,
        'print_interval': 40,
    },
    'strong': {
        'max_queries': None,
        'n_samples': 100,
        'k': 5,
        'print_interval': 100,
    },
}

# Change only this line when moving to a stronger machine
profile_name = 'balanced'
profile = profiles[profile_name]

cfg = ExperimentConfig(
    root=ROOT,
    python_exe=PY,
    generator_name='flanT5Small',
    lamp_num=4,
    retriever_name='bm25',
    alphas=(1, 2, 4, 8),
    max_queries=profile['max_queries'],
    n_samples=profile['n_samples'],
    k=profile['k'],
    remove_temp_files=True,
    skip_existing=True,
    mmr_base_retriever='bm25',
    mmr_lambda=0.6,
    run_tag=profile_name,
    print_interval=profile['print_interval'],
 )

ensure_paths_exist([cfg.python_exe])

print('Root   :', ROOT)
print('Python :', cfg.python_exe)
print('Profile:', profile_name)
print('Config :', cfg)


Root   : /home/student/Fair-RAG
Python : /anaconda/envs/azureml_py310_sdkv2/bin/python3.10
Profile: balanced
Config : ExperimentConfig(root=PosixPath('/home/student/Fair-RAG'), python_exe=PosixPath('/anaconda/envs/azureml_py310_sdkv2/bin/python3.10'), generator_name='flanT5Small', lamp_num=4, retriever_name='bm25', alphas=(1, 2, 4, 8), max_queries=833, n_samples=12, k=5, remove_temp_files=True, skip_existing=True, mmr_base_retriever='bm25', mmr_lambda=0.6, run_tag='balanced')


## Run Controls
Toggle stages independently. Baselines run first (deterministic BM25 + deterministic MMR).

In [8]:
RUN_GOLD = True
RUN_DETERMINISTIC_REF = True
RUN_MMR_DETERMINISTIC = True
RUN_MMR_LAMBDA_SWEEP = True
RUN_BASELINE_SUMMARY = True

RUN_BM25_GRID = True
RUN_NORMALIZE_EU = True
RUN_SANITY = True
RUN_ANALYSIS = True

# Deterministic references use one alpha only; value is for naming consistency.
DETERMINISTIC_ALPHA = 1
MMR_LAMBDAS = (0.6,)

# Sanity checks: oracle + stochastic BM25 across alphas on a small subset.
SANITY_MAX_QUERIES = 12
SANITY_N_SAMPLES = 10
SANITY_ALPHAS = (1, 2, 4, 8)
SANITY_GOLD_ALPHA = 8
SANITY_OUTPUT_SUFFIX = '_sanity'

# IMPORTANT for long runs on unstable servers:
# Keep this False so completed outputs are preserved and reruns resume safely.
FORCE_FRESH_RUN = False

In [9]:
import time
from datetime import datetime


def run_stage(name, enabled, fn):
    if not enabled:
        print(f'[skip] {name}')
        return
    print(f"\n=== START {name} @ {datetime.now().strftime('%H:%M:%S')} ===")
    t0 = time.time()
    fn()
    dt_sec = time.time() - t0
    print(f'=== DONE {name} in {dt_sec/60:.2f} min ===')


def build_sanity_cfg():
    return ExperimentConfig(
        root=cfg.root,
        python_exe=cfg.python_exe,
        generator_name=cfg.generator_name,
        lamp_num=cfg.lamp_num,
        retriever_name=cfg.retriever_name,
        alphas=SANITY_ALPHAS,
        max_queries=SANITY_MAX_QUERIES,
        n_samples=SANITY_N_SAMPLES,
        k=cfg.k,
        remove_temp_files=cfg.remove_temp_files,
        skip_existing=False,
        mmr_base_retriever=cfg.mmr_base_retriever,
        mmr_lambda=cfg.mmr_lambda,
    )


def build_cfg_with_mmr_lambda(mmr_lambda):
    return ExperimentConfig(
        root=cfg.root,
        python_exe=cfg.python_exe,
        generator_name=cfg.generator_name,
        lamp_num=cfg.lamp_num,
        retriever_name=cfg.retriever_name,
        alphas=cfg.alphas,
        max_queries=cfg.max_queries,
        n_samples=cfg.n_samples,
        k=cfg.k,
        remove_temp_files=cfg.remove_temp_files,
        skip_existing=cfg.skip_existing,
        mmr_base_retriever=cfg.mmr_base_retriever,
        mmr_lambda=mmr_lambda,
    )


def mmr_lambda_suffix(mmr_lambda):
    return f"_mmr_lambda_{mmr_lambda:.1f}".replace('.', 'p')


def run_sanity_checks():
    sanity_cfg = build_sanity_cfg()
    print(
        'Sanity config:',
        {
            'max_queries': sanity_cfg.max_queries,
            'n_samples': sanity_cfg.n_samples,
            'alphas': sanity_cfg.alphas,
            'suffix': SANITY_OUTPUT_SUFFIX,
        },
    )

    run_experiment_for_alpha(
        sanity_cfg,
        retriever_name='gold',
        alpha=SANITY_GOLD_ALPHA,
        output_suffix=SANITY_OUTPUT_SUFFIX,
    )

    for alpha in sanity_cfg.alphas:
        run_experiment_for_alpha(
            sanity_cfg,
            retriever_name=sanity_cfg.retriever_name,
            alpha=alpha,
            output_suffix=SANITY_OUTPUT_SUFFIX,
        )

    df_st = load_raw_rows(
        sanity_cfg,
        sanity_cfg.retriever_name,
        sanity_cfg.alphas,
        output_suffix=SANITY_OUTPUT_SUFFIX,
    )
    df_oracle = load_raw_rows(
        sanity_cfg,
        'gold',
        (SANITY_GOLD_ALPHA,),
        output_suffix=SANITY_OUTPUT_SUFFIX,
    )

    st_summary = (
        df_st.groupby('alpha', as_index=False)
        .agg(
            n_qids=('qid', 'nunique'),
            mean_ee_d=('ee_d', 'mean'),
            mean_ee_r=('ee_r', 'mean'),
            mean_eu=('eu', 'mean'),
        )
        .sort_values('alpha')
        .reset_index(drop=True)
    )

    oracle_summary = pd.DataFrame(
        [
            {
                'alpha': f'gold@{SANITY_GOLD_ALPHA}',
                'n_qids': df_oracle['qid'].nunique(),
                'mean_ee_d': df_oracle['ee_d'].mean(),
                'mean_ee_r': df_oracle['ee_r'].mean(),
                'mean_eu': df_oracle['eu'].mean(),
            }
        ]
    )

    print('\nSanity summary: stochastic BM25 by alpha')
    display(st_summary)
    print('Sanity summary: oracle gold anchor')
    display(oracle_summary)


def run_mmr_lambda_sweep():
    for mmr_lambda in MMR_LAMBDAS:
        mmr_cfg = build_cfg_with_mmr_lambda(mmr_lambda)
        run_mmr_deterministic(
            mmr_cfg,
            alpha=DETERMINISTIC_ALPHA,
            output_suffix=mmr_lambda_suffix(mmr_lambda),
        )


def load_mmr_lambda_sweep_rows():
    rows = []
    for mmr_lambda in MMR_LAMBDAS:
        df_lambda = load_raw_rows(
            cfg,
            'mmr',
            (DETERMINISTIC_ALPHA,),
            output_suffix=mmr_lambda_suffix(mmr_lambda),
        )
        rows.append(
            {
                'mmr_lambda': mmr_lambda,
                'n_qids': df_lambda['qid'].nunique(),
                'mean_ee_r': df_lambda['ee_r'].mean(),
                'mean_eu': df_lambda['eu'].mean(),
            }
        )
    return pd.DataFrame(rows).sort_values('mmr_lambda').reset_index(drop=True)


def show_deterministic_baselines():
    df_bm25 = load_raw_rows(
        cfg,
        cfg.retriever_name,
        (DETERMINISTIC_ALPHA,),
        output_suffix='_deterministic',
    )
    df_mmr = load_raw_rows(
        cfg,
        'mmr',
        (DETERMINISTIC_ALPHA,),
        output_suffix='_mmr_deterministic',
    )

    baseline_summary = pd.DataFrame(
        [
            {
                'retriever': f'{cfg.retriever_name}_deterministic',
                'alpha': DETERMINISTIC_ALPHA,
                'n_qids': df_bm25['qid'].nunique(),
                'mean_ee_r': df_bm25['ee_r'].mean(),
                'mean_eu': df_bm25['eu'].mean(),
            },
            {
                'retriever': 'mmr_deterministic',
                'alpha': DETERMINISTIC_ALPHA,
                'n_qids': df_mmr['qid'].nunique(),
                'mean_ee_r': df_mmr['ee_r'].mean(),
                'mean_eu': df_mmr['eu'].mean(),
            },
        ]
    )

    print('\nDeterministic baselines (raw EU and EE-R):')
    display(baseline_summary)


def show_mmr_lambda_sweep():
    sweep_summary = load_mmr_lambda_sweep_rows()
    print('\nDeterministic MMR lambda sweep (raw EU and EE-R):')
    display(sweep_summary)


if FORCE_FRESH_RUN:
    run_stage('reset outputs', True, lambda: reset_run_outputs(cfg, include_gold=True))

# Baselines first
run_stage('gold baseline (alpha=8)', RUN_GOLD, lambda: run_gold_baseline(cfg, alpha=8))
run_stage(
    'deterministic BM25 baseline',
    RUN_DETERMINISTIC_REF,
    lambda: run_deterministic_reference(
        cfg, alpha=DETERMINISTIC_ALPHA, output_suffix='_deterministic'
    ),
)
run_stage(
    'deterministic MMR baseline',
    RUN_MMR_DETERMINISTIC,
    lambda: run_mmr_deterministic(
        cfg, alpha=DETERMINISTIC_ALPHA, output_suffix='_mmr_deterministic'
    ),
)
run_stage('MMR lambda sweep (0.0 to 1.0)', RUN_MMR_LAMBDA_SWEEP, run_mmr_lambda_sweep)
run_stage('baseline summary (EU, EE-R)', RUN_BASELINE_SUMMARY, show_deterministic_baselines)
run_stage('MMR lambda sweep summary', RUN_MMR_LAMBDA_SWEEP, show_mmr_lambda_sweep)

# Full BM25 experiment stages
run_stage('BM25 alpha grid', RUN_BM25_GRID, lambda: run_retriever_grid(cfg))

def _normalize_stage():
    assert_consistent_qids_across_alphas(cfg, cfg.retriever_name)
    normalize_eu_grid(cfg)

run_stage('EU normalization (BM25 grid)', RUN_NORMALIZE_EU, _normalize_stage)
run_stage('sanity checks (oracle + stochastic alpha sweep)', RUN_SANITY, run_sanity_checks)


=== START gold baseline (alpha=8) @ 16:50:41 ===
[skip] existing: /home/student/Fair-RAG/experiment_results/flanT5Small/lamp4/gold/alpha_8.json
=== DONE gold baseline (alpha=8) in 0.00 min ===

=== START deterministic BM25 baseline @ 16:50:41 ===
[stale] alpha_1_deterministic.json has 30 queries but 833 requested — rerunning

> /anaconda/envs/azureml_py310_sdkv2/bin/python3.10 experiment.py --generator_name flanT5Small --lamp_num 4 --retriever_name bm25 --alpha 1 --k 5 --n_samples 1 --deterministic_ranking --output_suffix _deterministic --max_queries 833 --remove_temp_files --run_tag balanced
[run-log] /home/student/Fair-RAG/experiment_results/runs/20260326_165044_bm25_alpha1_deterministic_flanT5Small_lamp4_nq833_balanced
[10/833] avg EE-D: 1.0000 | avg EE-R: 0.1400 | avg EU (rouge-l): 0.0142
[20/833] avg EE-D: 1.0000 | avg EE-R: 0.1340 | avg EU (rouge-l): 0.0207
[30/833] avg EE-D: 1.0000 | avg EE-R: 0.1827 | avg EU (rouge-l): 0.0191
[40/833] avg EE-D: 1.0000 | avg EE-R: 0.1970 | avg 

KeyboardInterrupt: 

## Baseline-First Results, MMR Lambda Sweep, and BM25 Grid Analysis

In [ ]:
if RUN_ANALYSIS:
    # 1) Deterministic baseline comparison first (raw EU/EE-R).
    df_det = load_raw_rows(
        cfg,
        cfg.retriever_name,
        (DETERMINISTIC_ALPHA,),
        output_suffix='_deterministic',
    )
    df_mmr = load_raw_rows(
        cfg,
        'mmr',
        (DETERMINISTIC_ALPHA,),
        output_suffix='_mmr_deterministic',
    )

    baseline_compare = pd.DataFrame(
        [
            {
                'retriever': f'{cfg.retriever_name}_deterministic',
                'alpha': DETERMINISTIC_ALPHA,
                'n_qids': df_det['qid'].nunique(),
                'mean_ee_r': df_det['ee_r'].mean(),
                'mean_eu': df_det['eu'].mean(),
            },
            {
                'retriever': 'mmr_deterministic',
                'alpha': DETERMINISTIC_ALPHA,
                'n_qids': df_mmr['qid'].nunique(),
                'mean_ee_r': df_mmr['ee_r'].mean(),
                'mean_eu': df_mmr['eu'].mean(),
            },
        ]
    )
    print('Deterministic baselines first (raw EU and EE-R):')
    display(baseline_compare)

    # 2) Deterministic MMR lambda sweep before stochastic BM25 experiments.
    mmr_lambda_summary = load_mmr_lambda_sweep_rows()
    print('\nDeterministic MMR lambda sweep (raw EU and EE-R):')
    display(mmr_lambda_summary)

    # 3) BM25 normalized alpha-grid interval report.
    df = load_normalized_rows(cfg)
    df = add_ee_d_interval_bins(df)
    summary = summarize_by_interval(df)

    print(
        f'\nAnalyzing normalized BM25 alpha-grid for retriever: {cfg.retriever_name} '
        '(oracle gold excluded; used only as normalization reference)'
    )
    print('Per-query rows (BM25 alpha grid):', len(df))
    display(summary)

    # Deterministic interval views on raw scale.
    df_det_binned = add_ee_d_interval_bins(df_det)
    summary_det = summarize_by_interval(df_det_binned)
    print('\nDeterministic BM25 reference (raw EE/EU):')
    display(summary_det)

    df_mmr_binned = add_ee_d_interval_bins(df_mmr)
    summary_mmr = summarize_by_interval(df_mmr_binned)
    print('\nDeterministic MMR reference (raw EE/EU):')
    display(summary_mmr)

    out_name = f'notebook_outputs_{profile_name}'
    summary_fp = save_interval_outputs(cfg, summary, out_name)
    perq_fp = save_per_query_log(cfg, df, out_name)

    # Significance: MMR deterministic vs BM25 deterministic on raw deterministic metrics.
    pvals_ee_r = interval_pvalues_vs_deterministic(df_mmr_binned, df_det_binned, metric='ee_r')
    pvals_eu = interval_pvalues_vs_deterministic(df_mmr_binned, df_det_binned, metric='eu')
    pvals = pd.concat([pvals_ee_r, pvals_eu], ignore_index=True)

    pvals = pvals[pvals['n_qids'] > 0].copy()
    pvals = (
        pvals[[
            'ee_d_interval',
            'metric',
            'n_qids',
            'p_two_sided',
        ]]
        .sort_values(['ee_d_interval', 'metric'])
        .reset_index(drop=True)
    )
    display(pvals)
    pvals_fp = save_pvalue_outputs(cfg, pvals, out_name)

    print('Saved:')
    print(' -', summary_fp)
    print(' -', perq_fp)
    print(' -', pvals_fp)
else:
    print('Skipping analysis')

Deterministic baselines first (raw EU and EE-R):


,retriever,alpha,n_qids,mean_ee_r,mean_eu
0,bm25_deterministic,1,30,0.182695,0.019068
1,mmr_deterministic,1,30,0.189362,0.024094



Deterministic MMR lambda sweep (raw EU and EE-R):


,mmr_lambda,n_qids,mean_ee_r,mean_eu
0,0.6,30,0.176028,0.025545



Analyzing normalized BM25 alpha-grid for retriever: bm25 (oracle gold excluded; used only as normalization reference)
Per-query rows (BM25 alpha grid): 120


...


Deterministic MMR reference (raw EE/EU):


/home/student/Fair-RAG/notebook_experiment_utils.py:510: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_binned.groupby("ee_d_interval", dropna=False)


,ee_d_interval,n_queries,mean_eu,mean_ee_d,mean_ee_r
0,0.0-0.2,0,NaN,NaN,NaN
1,0.2-0.4,0,NaN,NaN,NaN
2,0.4-0.6,0,NaN,NaN,NaN
3,0.6-0.8,0,NaN,NaN,NaN
4,0.8-1.0,30,0.024094,1.0,0.189362


,ee_d_interval,metric,n_qids,p_two_sided
0,0.8-1.0,ee_r,30,0.317311
1,0.8-1.0,eu,30,0.248864


Saved:
 - /home/student/Fair-RAG/experiment_results/flanT5Small/lamp4/bm25/notebook_outputs_balanced/ee_d_interval_summary.csv
 - /home/student/Fair-RAG/experiment_results/flanT5Small/lamp4/bm25/notebook_outputs_balanced/per_query_all_metrics.csv
 - /home/student/Fair-RAG/experiment_results/flanT5Small/lamp4/bm25/notebook_outputs_balanced/pvalues_vs_deterministic_by_interval.csv


In [ ]:
# Force a true balanced rerun (833 queries) instead of reusing existing weak outputs.
cfg.skip_existing = False
FORCE_FRESH_RUN = True

print('Forced rerun settings:')
print(' profile_name      =', profile_name)
print(' cfg.max_queries   =', cfg.max_queries)
print(' cfg.skip_existing =', cfg.skip_existing)
print(' FORCE_FRESH_RUN   =', FORCE_FRESH_RUN)
print(' MMR_LAMBDAS       =', MMR_LAMBDAS)

Current deterministic MMR lambda sweep summary (raw EU and EE-R):


,mmr_lambda,n_qids,mean_ee_r,mean_eu
0,0.0,30,0.113333,0.015120
1,0.1,30,0.133333,0.019047
2,0.2,30,0.173333,0.013905
3,0.3,30,0.169362,0.020878
4,0.4,30,0.169362,0.025200
5,0.5,30,0.176028,0.022288
6,0.6,30,0.176028,0.025545
7,0.7,30,0.189362,0.024094
8,0.8,30,0.182695,0.021256
9,0.9,30,0.182695,0.021184


In [1]:
# ===== Run Status Inspector =====
# Shows the latest N runs: status (completed / interrupted), config, and metrics.
#
# To RESUME an interrupted run:
#   1. Make sure FORCE_FRESH_RUN = False (Cell 4)
#   2. Re-run Cells 1-2 (imports) and Cell 5 (pipeline)
#   The checkpoint is auto-detected and the run continues from where it left off.

from pathlib import Path
import json
import pandas as pd

# ---------- Config ----------
N_RECENT_RUNS = 5      # how many recent runs to list (newest first); 1 = latest only
RUN_TAG_FILTER = None  # narrow by tag, e.g. "balanced", "weak", or None for all
# ----------------------------

try:
    _runs_dir = ROOT / "experiment_results" / "runs"
except NameError:
    _runs_dir = Path("experiment_results/runs").resolve()

if not _runs_dir.exists() or not any(_runs_dir.iterdir()):
    print("No runs found yet. Run the pipeline (Cell 5) first.")
else:
    _all_runs = sorted(
        [d for d in _runs_dir.iterdir() if d.is_dir()],
        key=lambda d: d.name,
        reverse=True,
    )
    if RUN_TAG_FILTER:
        _all_runs = [d for d in _all_runs if RUN_TAG_FILTER in d.name]

    _runs_to_show = _all_runs[:N_RECENT_RUNS]
    print(f"Total runs found : {len(_all_runs)}")
    if RUN_TAG_FILTER:
        print(f"Tag filter       : '{RUN_TAG_FILTER}'")
    print(f"Showing latest   : {len(_runs_to_show)}\n")

    for _run_dir in _runs_to_show:
        _params_fp  = _run_dir / "params.json"
        _progress_fp = _run_dir / "progress.csv"

        if not _params_fp.exists():
            print(f"  {_run_dir.name}  [no params.json — skipping]\n")
            continue

        _p       = json.loads(_params_fp.read_text())
        _is_done = bool(_p.get("completed_at"))
        _status  = "COMPLETED" if _is_done else "INTERRUPTED / IN PROGRESS"

        print("=" * 70)
        print(f"  {_run_dir.name}")
        print(f"  Status    : {_status}")
        print(f"  Started   : {_p.get('started_at', '?')}")
        if _is_done:
            print(f"  Finished  : {_p.get('completed_at', '?')}")
        print(
            f"  Config    : retriever={_p.get('retriever','?')} | alpha={_p.get('alpha','?')}"
            f" | suffix={_p.get('output_suffix') or '(none)'}"
            f" | n_samples={_p.get('n_samples','?')}"
            f" | max_queries={_p.get('max_queries','?')}"
            f" | tag={_p.get('run_tag') or '(none)'}"
        )
        if _p.get("mmr_lambda") is not None:
            print(f"  MMR lambda: {_p['mmr_lambda']}")

        # --- Progress ---
        if _progress_fp.exists():
            _df = pd.read_csv(_progress_fp)
            if not _df.empty:
                _last      = _df.iloc[-1]
                _processed = int(_last["query_count"])
                _total_raw = _last["total_queries"]
                try:
                    _total_int = int(_total_raw)
                    _total_str = str(_total_int)
                    _pct_str   = f"{100.0 * _processed / _total_int:.0f}%"
                except (ValueError, TypeError):
                    _total_str = "?"
                    _pct_str   = "?"
                print(f"  Progress  : {_processed} / {_total_str} queries ({_pct_str})")
                print(
                    f"  Last avg  :"
                    f"  EE-D={float(_last['mean_ee_d']):.4f}"
                    f"  EE-R={float(_last['mean_ee_r']):.4f}"
                    f"  EU={float(_last['mean_eu']):.4f}"
                )
                if len(_df) > 1:
                    _disp = _df[["timestamp", "query_count", "total_queries",
                                 "mean_ee_d", "mean_ee_r", "mean_eu"]].copy()
                    _disp.columns = ["timestamp", "q_done", "q_total", "EE-D", "EE-R", "EU"]
                    print(f"\n  Progress log ({len(_df)} checkpoints):")
                    print(_disp.to_string(index=False, float_format="{:.4f}".format))
            else:
                print("  Progress  : progress.csv is empty")
        else:
            print("  Progress  : no progress.csv (run killed before first checkpoint)")

        # --- Output / checkpoint files ---
        _out_fp_str = _p.get("output_file")
        if _out_fp_str:
            _out_fp  = Path(_out_fp_str)
            _ckpt_fp = Path(_out_fp_str[:-5] + "_ckpt.json")
            print(f"  Output    : {'EXISTS' if _out_fp.exists() else 'MISSING'} — {_out_fp.name}")
            print(
                f"  Checkpoint: {'EXISTS (resume ready)' if _ckpt_fp.exists() else 'not present'}"
                f" — {_ckpt_fp.name}"
            )

        # --- Resume / restart guidance ---
        if not _is_done:
            print()
            if _out_fp_str:
                _ckpt_fp = Path(_out_fp_str[:-5] + "_ckpt.json")
                if _ckpt_fp.exists():
                    _ckpt_count = len(json.loads(_ckpt_fp.read_text()).get("results", {}))
                    print(f"  >> RESUME : checkpoint has {_ckpt_count} queries saved.")
                    print(f"     Keep FORCE_FRESH_RUN = False, then re-run Cells 1-2 and Cell 5.")
                    print(f"     The run will skip {_ckpt_count} already-processed queries.")
                else:
                    print(f"  >> RESTART: no checkpoint found — run will start from scratch.")
                    print(f"     Keep FORCE_FRESH_RUN = False and re-run Cells 1-2 and Cell 5.")

        print()


Total runs found : 1
Showing latest   : 1

  20260325_212333_gold_alpha8_plain_flanT5Small_lamp4_nq833_balanced
  Status    : INTERRUPTED / IN PROGRESS
  Started   : 2026-03-25T21:23:33
  Config    : retriever=gold | alpha=8 | suffix=(none) | n_samples=12 | max_queries=833 | tag=balanced
  Progress  : 680 / 833 queries (82%)
  Last avg  :  EE-D=0.3847  EE-R=0.9870  EU=0.0751

  Progress log (68 checkpoints):
          timestamp  q_done  q_total   EE-D   EE-R     EU
2026-03-25T21:27:16      10      833 0.5269 0.9921 0.0476
2026-03-25T21:30:36      20      833 0.4893 0.9922 0.0627
2026-03-25T21:33:54      30      833 0.4406 0.9944 0.0623
2026-03-25T21:36:55      40      833 0.4135 0.9956 0.0651
2026-03-25T21:39:58      50      833 0.3862 0.9965 0.0665
2026-03-25T21:42:35      60      833 0.3821 0.9944 0.0701
2026-03-25T21:45:54      70      833 0.3978 0.9948 0.0703
2026-03-25T21:49:23      80      833 0.4083 0.9948 0.0673
2026-03-25T21:53:02      90      833 0.4304 0.9902 0.0663
2026-03-